In [9]:
import pandas as pd
import matplotlib.pyplot as plt
from epiweeks import Week

d = pd.read_csv("./data/PA_ILI.csv")

d['Date'] = d.apply(
    lambda row: Week(row['YEAR'], row['WEEK']).startdate(),
    axis=1
)
d['Date'] = pd.to_datetime(d['Date'])

d_monthly = (
    d
    .set_index('Date')
    .resample('ME')
    .agg({
        'ILITOTAL': 'sum',     # numerator
        'TOTAL PATIENTS': 'sum'  # denominator
    })
    .reset_index()
)

d_monthly['ILI'] = d_monthly['ILITOTAL'] / d_monthly['TOTAL PATIENTS']
print(d_monthly)

ModuleNotFoundError: No module named 'epiweeks'

In [ ]:

vax = pd.read_csv("./data/Influenza_Vaccination_Coverage_PA.csv")
vax_2010_2025 = vax.loc[6: ] #selects only surveys after 2010
vax_2010_2025.rename(columns={'Season/Survey Year': 'Year'}, inplace=True) 
vax_2010_2025['YearNumeric'] = vax_2010_2025['Year'].str.split('-').str[0].astype(int)
vax_2010_2025['year'] = vax_2010_2025['YearNumeric']
vax_2010_2025['month'] = vax_2010_2025['Month']

vax_2010_2025['Date'] = pd.to_datetime(vax_2010_2025[['year', 'month']].assign(day=1))
print(vax_2010_2025.head())
print(d_monthly.head())
fig, ax = plt.subplots() 
plt.plot (vax_2010_2025["Date"],vax_2010_2025["New_per_Month"])

fig, ax = plt.subplots()
plt.plot (d_monthly['Date'], d_monthly['ILI'])

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(12,5))
ax1.plot(
    d_monthly["Date"],
    d_monthly["ILI"],
    color="crimson",
    linewidth=2,
    label="ILI"
)
ax1.set_ylabel("ILI Rate", color="crimson")
ax1.tick_params(axis='y', labelcolor='crimson')

ax2 = ax1.twinx()
ax2.plot(
    vax_2010_2025["Date"],
    vax_2010_2025["New_per_Month"],
    color="navy",
    linewidth=2,
    label="Vaccinations"
)
ax2.set_ylabel("Vaccinations", color="navy")
ax2.tick_params(axis='y', labelcolor='navy')

# --- Formatting ---
ax1.set_xlabel("Date")
fig.suptitle("ILI vs Vaccination Over Time", fontsize=14)

fig.autofmt_xdate()
fig.tight_layout()

plt.show()

In [ ]:
import seaborn as sns

vax_2 = vax_2010_2025.iloc[7:]
vax_2 = vax_2.reset_index(drop=True)

ili_2 = d_monthly.iloc[:-15]
ili_2 = ili_2.reset_index(drop=True)

vax_2['year_month'] = vax_2['Date'].dt.strftime('%Y-%m')
ili_2['year_month'] = ili_2['Date'].dt.strftime('%Y-%m')

vax_ili= pd.merge(vax_2, ili_2, on= 'year_month')
print(vax_ili)


sns.residplot(x = 'New_per_Month',y = 'ILI', data = vax_ili, color="purple")
plt.show()
